# Expected Agents

Runs the Tideway expected agents report.


## Setup

Set `APPLIANCE_NAME` or `APPLIANCE_INDEX` when `config.yaml` contains multiple appliances. Set `WRITE_OUTPUT = True` to write Tideway's canonical report file.

In [ ]:
from pathlib import Path
from xml.etree import ElementTree

import pandas as pd
import tideway
from tideway import notebooks as tw_nb

APPLIANCE_NAME = None
APPLIANCE_INDEX = 0
WRITE_OUTPUT = False
OUTPUT_BASE_DIR = None

tw = tw_nb.appliance_from_config(appliance_name=APPLIANCE_NAME, appliance_index=APPLIANCE_INDEX)
output_dir = tw_nb.output_dir_for(tw.target, OUTPUT_BASE_DIR)
print("Target      :", tw.target)
print("Output dir  :", output_dir)


## Run Report

In [ ]:
REPORT_NAME = 'expected_agents'
REPORT_OPTIONS = {}

reports = tw.reports()
report_method = getattr(reports, REPORT_NAME)

if WRITE_OUTPUT:
    result = report_method(output_dir=str(output_dir), **REPORT_OPTIONS)
else:
    result = report_method(**REPORT_OPTIONS)

df = tw_nb.report_to_dataframe(result)
no_expected_agent_data = (
    len(df) == 1
    and 'Host Name' in df.columns
    and df.iloc[0]['Host Name'] == 'No data returned'
)

if no_expected_agent_data:
    print('No missing-agent exceptions returned.')
else:
    display(df.head(50))
    print("Rows:", len(df))

QUERY_TITLE = 'Expected Agents Coverage'
query_path = tw_nb.find_repo_root() / 'queries' / 'queries.xml'
query_root = ElementTree.parse(query_path).getroot()
query_node = next((node for node in query_root.findall('query') if node.get('title') == QUERY_TITLE), None)
if query_node is None or query_node.find('search') is None:
    raise ValueError(f'Query {QUERY_TITLE!r} was not found in {query_path}')
AGENT_COVERAGE_QUERY = ''.join(query_node.find('search').itertext()).strip()

coverage_columns = [
    'Hostname',
    'Windows Hosts',
    'SCCM',
    'Sophos AV',
    'Qualys Cloud Agent',
    'BMC Client Management',
    'McAfee Endpoint Security',
    'BMC Patrol Agent',
    'Symantec Endpoint Protection',
]
coverage_rows = tw.data().search({'query': AGENT_COVERAGE_QUERY}, format='object', limit=0)
coverage_df = pd.DataFrame(coverage_rows).reindex(columns=coverage_columns)

if coverage_df.empty:
    coverage_df = pd.DataFrame([[
        'No Windows hosts',
        *([0] * (len(coverage_columns) - 1)),
    ]], columns=coverage_columns)
else:
    for column in coverage_columns[1:]:
        coverage_df[column] = pd.to_numeric(coverage_df[column], errors='coerce').fillna(0).astype(int)

print('Agent coverage:')
display(coverage_df.head(50))
if getattr(result, "files", None):
    print("Files:")
    for path in result.files:
        print(" -", path)
